In [3]:
import requests              
from bs4 import BeautifulSoup
import pandas as pd          
import time                  

Step 1: Data scraping

In [4]:
#urls
base_url = "https://www.federalreserve.gov"
hist_url = "https://www.federalreserve.gov/monetarypolicy/materials/assets/final-hist.json"
recent_url= "https://www.federalreserve.gov/monetarypolicy/materials/assets/final-recent.json"

In [5]:
hist_data = requests.get(hist_url).json()
print(hist_data['mtgitems'][0])

{'d': '2010-01-27', 'mtg': 'January 27, 2010', 'type': 'Ag', 'name': 'Agenda', 'url': '/monetarypolicy/files/FOMC20100127Agenda.pdf', 'size': 14.86, 'dt': '2010-01-27'}


In [6]:
recent_data=requests.get(recent_url).json()
print(recent_data['mtgitems'][0])

{'d': '2025-08-22', 'mtg': 'August 22, 2025', 'type': 'Pl', 'name': 'Statement on Longer-Run Goals and Monetary Policy Strategy', 'url': '/newsevents/pressreleases/monetary20250822a.htm', 'dt': '2025-08-22'}


In [7]:
#list
hist_statements = [item for item in hist_data['mtgitems'] 
              if item['type'] == 'St' 
              and item['d'] >= '2008-01-01']

In [8]:
recent_statements= [item for item in recent_data['mtgitems']
                    if item['type']=='St']

In [9]:
print(hist_statements[0])

{'d': '2010-01-27', 'mtg': 'January 27, 2010', 'type': 'St', 'name': 'Statement', 'url': '/newsevents/press/monetary/20100127a.htm', 'dt': '2010-01-27'}


In [10]:
print(recent_statements[0])

{'d': '2025-09-17', 'mtg': 'September 16-17, 2025', 'type': 'St', 'dt': '2025-09-17', 'files': [{'name': 'HTML', 'url': '/newsevents/pressreleases/monetary20250917a.htm'}, {'name': 'PDF', 'url': '/monetarypolicy/files/monetary20250917a1.pdf'}]}


In [11]:
statements=hist_statements+recent_statements

In [12]:
print(f"Historical statemnts: {len(hist_statements)}")
print(f"Recent Statements: {len(recent_statements)}")
print(f"Total statements: {len(statements)}")

Historical statemnts: 28
Recent Statements: 124
Total statements: 152


In [12]:
results = []

for i, statement in enumerate(statements):
    # Handle both URL structures
    if 'url' in statement:
        full_url = base_url + statement['url']
    elif 'files' in statement:
        full_url = base_url + statement['files'][0]['url']
    else:
        continue
    
    try:
        response = requests.get(full_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        content_div = soup.find('div', class_='col-xs-12 col-sm-8 col-md-8')
        text = content_div.get_text(strip=True) if content_div else None
        
        results.append({
            'date': statement['d'],
            'meeting': statement['mtg'],
            'url': full_url,
            'text': text
        })
        
        print(f"[{i+1}/{len(statements)}] Scraped: {statement['d']}")
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error on {statement['d']}: {e}")

df = pd.DataFrame(results)
df.to_csv('data/raw/fomc_statements.csv', index=False)
print(f"\nSaved {len(df)} statements")

[1/152] Scraped: 2010-01-27
[2/152] Scraped: 2010-03-16
[3/152] Scraped: 2010-04-28
[4/152] Scraped: 2010-05-09
[5/152] Scraped: 2010-06-23
[6/152] Scraped: 2010-08-10
[7/152] Scraped: 2010-09-21
[8/152] Scraped: 2010-11-03
[9/152] Scraped: 2010-12-14
[10/152] Scraped: 2009-01-28
[11/152] Scraped: 2009-03-17
[12/152] Scraped: 2009-04-29
[13/152] Scraped: 2009-06-24
[14/152] Scraped: 2009-08-11
[15/152] Scraped: 2009-09-22
[16/152] Scraped: 2009-11-04
[17/152] Scraped: 2009-12-15
[18/152] Scraped: 2008-01-21
[19/152] Scraped: 2008-01-30
[20/152] Scraped: 2008-03-10
[21/152] Scraped: 2008-03-18
[22/152] Scraped: 2008-04-30
[23/152] Scraped: 2008-06-25
[24/152] Scraped: 2008-08-08
[25/152] Scraped: 2008-09-16
[26/152] Scraped: 2008-10-07
[27/152] Scraped: 2008-10-29
[28/152] Scraped: 2008-12-16
[29/152] Scraped: 2025-09-17
[30/152] Scraped: 2025-10-29
[31/152] Scraped: 2025-12-10
[32/152] Scraped: 2026-01-28
[33/152] Scraped: 2025-05-07
[34/152] Scraped: 2025-06-18
[35/152] Scraped: 2025-

In [13]:
df_statements=pd.read_csv('data/raw/fomc_statements.csv')

In [14]:
print(df_statements.isnull().sum())

date       0
meeting    0
url        0
text       0
dtype: int64


In [15]:
#Scrapping the minutes
hist_minutes=[item for item in hist_data['mtgitems']
            if item['type']=='Mn'
            and item['d']>='2008-01-01'              
              ]
recent_minutes =[item for item in recent_data['mtgitems']
                 if item['type']=='Mn']

In [16]:
print(recent_minutes[0])

{'d': '2025-09-17', 'mtg': 'September 16-17, 2025', 'type': 'Mn', 'dt': '2025-10-08', 'files': [{'name': 'HTML', 'url': '/monetarypolicy/fomcminutes20250917.htm'}, {'name': 'PDF', 'url': '/monetarypolicy/files/fomcminutes20250917.pdf'}]}


In [17]:
minutes=hist_minutes+recent_minutes
print(f"Historical minutes: {len(hist_minutes)}")
print(f"Recent minutes: {len(recent_minutes)}")
print(f"Total minutes: {len(minutes)}")

Historical minutes: 35
Recent minutes: 126
Total minutes: 161


In [23]:
results = []

for i, statement in enumerate(minutes):
    if 'url' in statement:
        full_url = base_url + statement['url']
    elif 'files' in statement:
        full_url = base_url + statement['files'][0]['url']
    else:
        continue
    
    try:
        response = requests.get(full_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        content_div = soup.find('div', class_='col-xs-12 col-sm-8 col-md-8')
        text = content_div.get_text(strip=True) if content_div else None
        
        results.append({
            'date': statement['d'],
            'meeting': statement['mtg'],
            'url': full_url,
            'text': text
        })
        
        print(f"[{i+1}/{len(minutes)}] Scraped: {statement['d']}")
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error on {statement['d']}: {e}")

df_minutes = pd.DataFrame(results)
df_minutes.to_csv('data/raw/fomc_minutes.csv', index=False)
print(f"\nDone! Saved {len(df_minutes)} minutes")

[1/161] Scraped: 2010-01-27
[2/161] Scraped: 2010-03-16
[3/161] Scraped: 2010-04-28
[4/161] Scraped: 2010-05-09
[5/161] Scraped: 2010-06-23
[6/161] Scraped: 2010-08-10
[7/161] Scraped: 2010-09-21
[8/161] Scraped: 2010-10-15
[9/161] Scraped: 2010-11-03
[10/161] Scraped: 2010-12-14
[11/161] Scraped: 2009-01-16
[12/161] Scraped: 2009-01-28
[13/161] Scraped: 2009-02-07
[14/161] Scraped: 2009-03-17
[15/161] Scraped: 2009-04-29
[16/161] Scraped: 2009-06-03
[17/161] Scraped: 2009-06-24
[18/161] Scraped: 2009-08-11
[19/161] Scraped: 2009-09-22
[20/161] Scraped: 2009-11-04
[21/161] Scraped: 2009-12-15
[22/161] Scraped: 2008-01-09
[23/161] Scraped: 2008-01-21
[24/161] Scraped: 2008-01-30
[25/161] Scraped: 2008-03-10
[26/161] Scraped: 2008-03-18
[27/161] Scraped: 2008-04-30
[28/161] Scraped: 2008-06-25
[29/161] Scraped: 2008-07-24
[30/161] Scraped: 2008-08-08
[31/161] Scraped: 2008-09-16
[32/161] Scraped: 2008-09-29
[33/161] Scraped: 2008-10-07
[34/161] Scraped: 2008-10-29
[35/161] Scraped: 2008-

In [18]:
df_minutes=pd.read_csv("data/raw/fomc_minutes.csv")

In [19]:
#Scraping fed chair speeches

response = requests.get("https://www.federalreserve.gov/json/ne-speeches.json")
response.encoding = 'utf-8-sig'
speeches_data = response.json()
print(type(speeches_data))
print(speeches_data[:2])

<class 'list'>
[{'d': '3/12/2026 11:00:00 AM', 't': 'Capital Rules for the Real Economy', 's': 'Vice Chair for Supervision Michelle W. Bowman', 'lo': 'At the Cato Institute Policy Forum: Basel III and Bank Capital Rules, Washington, D.C.', 'l': '/newsevents/speech/bowman20260312a.htm', 'a': '', 'o': 'no', 'v': 'https://www.cato.org/events/basel-iii-bank-capital-rules-conversation-vice-chair-michelle-bowman ', 'video': 'No'}, {'d': '3/3/2026 8:30:00 AM', 't': 'Liquidity Resiliency, Financial Stability, and the Role of the Federal Reserve', 's': 'Vice Chair for Supervision Michelle W. Bowman', 'lo': 'At The Roundtable on Liquidity and Lender of Last Resort sponsored by the Committee on Capital Markets Regulation, Washington, D.C.', 'l': '/newsevents/speech/bowman20260303a.htm', 'a': '', 'o': 'no', 'v': '', 'video': 'No'}]


In [20]:
#total speeches count
speeches = [item for item in speeches_data if 'd' in item]
print(f"Total speeches: {len(speeches)}")
print(f"Most recent: {speeches[0]['d']}")
print(f"Oldest: {speeches[-1]['d']}")

Total speeches: 1293
Most recent: 3/12/2026 11:00:00 AM
Oldest: 1/7/2017 11:15:00 AM


In [21]:
#feds between 2017-2026
fed_chairs=['Powell', 'Yellen']
filtered_speeches=[
    item for item in speeches 
    if any(chair in item['s'] for chair in fed_chairs)
    and int(item['d'].split('/')[-1].split(' ')[0]) >= 2017
]
print(f"Total filtered speeches: {len(filtered_speeches)}")
print(filtered_speeches[0])

Total filtered speeches: 105
{'d': '1/11/2026 7:30:00 PM', 't': 'Statement from Federal Reserve Chair Jerome H. Powell', 's': 'Chair Jerome H. Powell', 'lo': '', 'l': '/newsevents/speech/powell20260111a.htm', 'a': '', 'o': 'no', 'v': '', 'video': 'Yes'}


In [36]:
results = []

for i, speech in enumerate(filtered_speeches):
    full_url = base_url + speech['l']
    
    try:
        response = requests.get(full_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        content_div = soup.find('div', class_='col-xs-12 col-sm-8 col-md-8')
        text = content_div.get_text(strip=True) if content_div else None
        
        results.append({
            'date': speech['d'],
            'title': speech['t'],
            'speaker': speech['s'],
            'url': full_url,
            'text': text
        })
        
        print(f"[{i+1}/{len(filtered_speeches)}] Scraped: {speech['d']} - {speech['t'][:50]}")
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error on {speech['d']}: {e}")

df_speeches = pd.DataFrame(results)
df_speeches.to_csv('data/raw/fed_speeches.csv', index=False)
print(f"\nDone! Saved {len(df_speeches)} speeches")

[1/105] Scraped: 1/11/2026 7:30:00 PM - Statement from Federal Reserve Chair Jerome H. Pow
[2/105] Scraped: 12/1/2025 8:00:00 PM - Opening Remarks
[3/105] Scraped: 10/14/2025 12:20:00 PM - Understanding the Fed’s Balance Sheet
[4/105] Scraped: 10/9/2025 8:30:00 AM - Welcoming Remarks
[5/105] Scraped: 9/23/2025 12:35:00 PM - Economic Outlook
[6/105] Scraped: 8/22/2025 10:00:00 AM - Monetary Policy and the Fed’s Framework Review
[7/105] Scraped: 7/22/2025 8:30:00 AM - Opening Remarks
[8/105] Scraped: 6/2/2025 1:00:00 PM - Opening Remarks
[9/105] Scraped: 5/25/2025 2:40:00 PM - Baccalaureate Remarks
[10/105] Scraped: 5/15/2025 8:40:00 AM - Opening Remarks
[11/105] Scraped: 4/16/2025 1:30:00 PM - Economic Outlook
[12/105] Scraped: 4/4/2025 11:25:00 AM - Economic Outlook
[13/105] Scraped: 3/7/2025 12:30:00 PM - Economic Outlook
[14/105] Scraped: 11/14/2024 3:00:00 PM - Economic Outlook
[15/105] Scraped: 9/30/2024 1:55:00 PM - Economic Outlook
[16/105] Scraped: 9/26/2024 9:20:00 AM - Opening

In [22]:
df_speeches=pd.read_csv('data/raw/fed_speeches.csv')

In [23]:
print(df_speeches.shape)
print(df_speeches['text'].isna().sum())

(105, 5)
0


In [ ]:
import yfinance as yf
#downloading stock market indexes
sp500 = yf.download('^GSPC', start='2008-01-01', end='2026-01-01')
vix = yf.download('^VIX', start='2008-01-01', end='2026-01-01')
treasury_10y = yf.download('^TNX', start='2008-01-01', end='2026-01-01')
treasury_2y = yf.download('^IRX', start='2008-01-01', end='2026-01-01')

print(sp500.shape)
print(sp500.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

(4529, 5)
Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
2008-01-02  1447.160034  1471.770020  1442.069946  1467.969971  3452650000
2008-01-03  1447.160034  1456.800049  1443.729980  1447.550049  3429500000
2008-01-04  1411.630005  1444.010010  1411.189941  1444.010010  4166000000
2008-01-07  1416.180054  1423.869995  1403.449951  1414.069946  4221260000
2008-01-08  1390.189941  1430.280029  1388.300049  1415.709961  4705390000


In [25]:
sp500.to_csv('data/raw/sp500.csv')
vix.to_csv('data/raw/vix.csv')
treasury_10y.to_csv('data/raw/treasury_10y.csv')
treasury_2y.to_csv('data/raw/treasury_2y.csv')

print("Market data saved!")

Market data saved!
